Class implementation for CAX simulation module

```26/05/2026```

In [1]:
import Shadow
import numpy as np
import matplotlib.pyplot as plt

What do I need to accomplish? I need to be able to simulate the passage of light through the CAX beamline, from source through slits to DVF screen. I then need to be able to extract the resulting image at a given $y$ position (optical axis). Then I need to perform movements to the mirror and regenerate the image.

To create a general `CAXSim` class, I'll first write a basic `OpticalElement` wrapper class for Shadow OEs, then I'll create children with more specific methods, all the way to a `BeamLine` class that possesses the necessary attributes and methods to perform simulations and modify the system. 

In [2]:
!which python

/home/amici/miniconda3/bin/python


In [2]:
from shadowpy.beamline import BeamLine

ModuleNotFoundError: No module named 'shadowpy'

In [5]:
from shadowpy.optical_elements import ToroidalMirror, Slits, Screen
from shadowpy.sources import BendingMagnet
from shadowpy.beamline import BeamLine
from shadowpy.utils import rotation_matrix, ReferenceFrame

ModuleNotFoundError: No module named 'shadowpy'

In [ ]:
def rotation_matrix(axis, angle):
    """
    Generate a rotation matrix for a given axis and angle.
    
    Parameters:
        axis (str): The axis of rotation ('x', 'y', or 'z').
        angle (float): The angle of rotation in degrees.
    
    Returns:
        np.ndarray: The 3x3 rotation matrix.
    """
    angle = np.radians(angle)
    c = np.cos(angle)
    s = np.sin(angle)

    if axis == 'x':
        return np.array([[1, 0, 0],
                         [0, c, -s],
                         [0, s, c]])
    elif axis == 'y':
        return np.array([[c, 0, s],
                         [0, 1, 0],
                         [-s, 0, c]])
    elif axis == 'z':
        return np.array([[c, -s, 0],
                         [s, c, 0],
                         [0, 0, 1]])
    else:
        raise ValueError("Axis must be 'x', 'y', or 'z'.")

In [ ]:
class ReferenceFrame:
    """
    Right-handed orthonormal reference frame.

    Convention:
        v_lab = R @ v_local

    Columns of R are the local basis vectors expressed
    in lab coordinates.
    """

    def __init__(self, origin=None, orientation=None, name=None):

        self.name = name or "frame"

        self.origin = np.array(
            origin if origin is not None else [0., 0., 0.],
            dtype=float
        )

        self.R = np.array(
            orientation if orientation is not None else np.eye(3),
            dtype=float
        )

        self._validate_rotation_matrix()

    # -------------------------------------------------
    # Validation
    # -------------------------------------------------

    def _validate_rotation_matrix(self, atol=1e-10):

        should_be_identity = self.R.T @ self.R

        if not np.allclose(should_be_identity, np.eye(3), atol=atol):
            raise ValueError("Orientation matrix is not orthogonal")

        det = np.linalg.det(self.R)

        if not np.isclose(det, 1.0, atol=atol):
            raise ValueError("Orientation matrix must have determinant +1")

    # -------------------------------------------------
    # Vector transforms
    # -------------------------------------------------

    def vector_to_lab(self, v_local):
        v_local = np.asarray(v_local, dtype=float)
        return self.R @ v_local

    def vector_from_lab(self, v_lab):
        v_lab = np.asarray(v_lab, dtype=float)
        return self.R.T @ v_lab

    # -------------------------------------------------
    # Point transforms
    # -------------------------------------------------

    def point_to_lab(self, p_local):
        p_local = np.asarray(p_local, dtype=float)
        return self.origin + self.R @ p_local

    def point_from_lab(self, p_lab):
        p_lab = np.asarray(p_lab, dtype=float)
        return self.R.T @ (p_lab - self.origin)

    # -------------------------------------------------
    # Relative frame construction
    # -------------------------------------------------

    def child_frame(self, relative_origin, relative_rotation, name=None):
        """
        Create a new frame defined relative to THIS frame.
        """

        relative_origin = np.asarray(relative_origin, dtype=float)
        relative_rotation = np.asarray(relative_rotation, dtype=float)

        new_origin = self.point_to_lab(relative_origin)
        new_rotation = self.R @ relative_rotation

        return ReferenceFrame(
            origin=new_origin,
            orientation=new_rotation,
            name=name
        )

    # -------------------------------------------------
    # Relations between frames
    # -------------------------------------------------

    def rotation_to(self, other):
        """
        Rotation matrix mapping vectors from this frame
        into another frame.
        """

        return other.R.T @ self.R

    def __repr__(self):
        return (
            f"ReferenceFrame(name={self.name}, "
            f"origin={self.origin})"
        )

In [ ]:
class OpticalElement:
    """
    A wrapper class for a Shadow optical element, 
    providing methods to load specifications and interact with the element.
    """

    def __init__(self, name: str, specification_file: str):
        self.name = name
        self.shadow_oe = Shadow.OE()
        self.specification_file = specification_file
        self.load_specification()
        # To be set when the element is added to a beamline
        self.frame = None  
        self.beamline = None
        # Flag to track if the element has been updated since last load
        self.updated = False  
        # Original properties to reset after simulation
        self.specification = self.shadow_oe.to_dictionary()
        # Image
        self.image = None
        # Persistent attributes that should not be reset after simulation
        self.persistent_attributes = []
        
    # General attributes    
        
    @property
    def orientation(self):
        """
        Optical element orientation angle (degrees) relative to the previous element.
        Positive values indicate counter-clockwise rotation.
        """
        return self.shadow_oe.ALPHA
    

    @property
    def source_distance(self):
        """
        Distance from the previous element to the source point (user units).
        """
        return self.shadow_oe.T_SOURCE
    
    @property 
    def image_distance(self):
        """
        Distance from the previous element to the image point (user units).
        """
        return self.shadow_oe.T_IMAGE 

    @property
    def units(self):
        """
        User units for distances.
        """

        return self.shadow_oe.unit()
    

    def load_specification(self, specification_file: str = None):
        """
        Load the optical element specification from a file.
        
        Parameters:
            specification_file (str): The path to the specification file. 
            If None, uses the default specification file.
        """

        if specification_file is None:
            specification_file = self.specification_file
        self.shadow_oe.load(specification_file)
        # self.specification = self.shadow_oe.to_dictionary()


class Source():
    """
    A wrapper class for a Shadow source optical element.
    """

    def __init__(self, name: str, specification_file: str):
        self.name = name
        self.shadow_oe = Shadow.Source()
        self.specification_file = specification_file
        self.load_specification()
        self.frame = None  # To be set when the element is added to a beamline
    
    def load_specification(self, specification_file: str = None):
        """
        Load the optical element specification from a file.
        
        Parameters:
            specification_file (str): The path to the specification file. 
            If None, uses the default specification file.
        """

        if specification_file is None:
            specification_file = self.specification_file
        self.shadow_oe.load(specification_file)

class BendingMagnet(Source):
    """
    A wrapper class for a Shadow bending magnet optical element.
    """

    def __init__(self, name: str, specification_file: str):
        super().__init__(name, specification_file)
        
        if self.shadow_oe.FDISTR != 4:
            raise ValueError(f"File {specification_file} is not"
                             f" configured for a synchrotron source.")

class ToroidalMirror(OpticalElement):
    """
    A wrapper class for a Shadow toroidal mirror optical element.
    """

    def __init__(self, name: str, specification_file: str):
        super().__init__(name, specification_file)
        
        if self.shadow_oe.FMIRR != 3:
            raise ValueError(f"File {specification_file} is not"
                             f" configured for a toroidal mirror: "
                             f"FMIRR = {self.shadow_oe.FMIRR}")


    @property
    def offset(self):
        """
        Get the current offset of the mirror's position in the global frame.
        
        Returns:
            np.ndarray: A 3D vector representing the offset in the global frame.
        """
        if self.frame is None:
            raise ValueError("Mirror frame is not defined. Please add the mirror to a beamline first.")
        
        offset_local = np.array([
            self.shadow_oe.OFFX,
            self.shadow_oe.OFFY,
            self.shadow_oe.OFFZ
        ])
        
        return self.frame.vector_to_lab(offset_local)
    
    @offset.setter
    def offset(self, offset_vector: np.ndarray):
        """
        Apply an offset to the mirror's position.
        
        Parameters:
            offset_vector (np.ndarray): A 3D vector representing the offset in the mirror's local frame.
        """
        if self.frame is None:
            raise ValueError("Mirror frame is not defined. Please add the mirror to a beamline first.")
        
        # Convert the offset vector from the mirror's local frame to the lab frame
        offset_local = self.frame.vector_from_lab(offset_vector)
        
        self.shadow_oe.OFFX = offset_local[0]
        self.shadow_oe.OFFY = offset_local[1]
        self.shadow_oe.OFFZ = offset_local[2]

    @property
    def tilt(self):
        """
        Get the current tilt angles of the mirror's orientation in the global frame.
        
        Returns:
            np.ndarray: A 3D vector representing the tilt angles (in degrees) around the lab frame axes.
        """
        if self.frame is None:
            raise ValueError("Mirror frame is not defined. Please add the mirror to a beamline first.")
        
        tilt_local = np.array([
            self.shadow_oe.X_ROT,
            self.shadow_oe.Y_ROT,
            self.shadow_oe.Z_ROT
        ])
        
        return self.frame.vector_to_lab(tilt_local)

    @tilt.setter
    def tilt(self, tilt_angles: np.ndarray):
        """
        Apply tilts to the mirror's orientation.
        
        Parameters:
            tilt_angles (np.ndarray): A 3D vector representing the tilt angles 
            (in degrees) around the lab frame axes.
        """
        if self.frame is None:
            raise ValueError("Mirror frame is not defined. Please add the mirror to a beamline first.")
        
        # Convert the tilt angles from the mirror's local frame to the lab frame
        self.tilt_global = tilt_angles
        self.tilt_local = self.frame.vector_from_lab(tilt_angles)
        
        self.shadow_oe.X_ROT = self.tilt_local[0]
        self.shadow_oe.Y_ROT = self.tilt_local[1]
        self.shadow_oe.Z_ROT = self.tilt_local[2]

class Screen(OpticalElement):
    """
    A wrapper class for a Shadow screen optical element.
    """

    def __init__(self, name: str, specification_file: str, screen_dimensions: tuple = (5., 5.)):
        super().__init__(name, specification_file)
        
        if self.shadow_oe.F_SCREEN != 1:
            raise ValueError(f"File {specification_file} is not"
                             f" configured for a screen: "
                             f"F_SCREEN = {self.shadow_oe.F_SCREEN}")

class Slits(Screen):
    """
    A wrapper class for a Shadow slits optical element.
    """

    def __init__(self, name: str, specification_file: str, slit_positions: tuple = (0., 0.)):
        super().__init__(name, specification_file)
        
        if any(self.shadow_oe.I_SLIT) != 1:
            raise ValueError(f"File {specification_file} is not"
                             f" configured for slits: "
                             f"I_SLIT = {self.shadow_oe.I_SLIT}")


In [ ]:
class BeamLine:
    """
    A class representing a beamline, which is a sequence of optical elements.
    """

    def __init__(self, name: str, beam: Shadow.Beam, source: Source, optical_elements: list = None, 
                 beamline_frame=None, lab_frame=None):
        self.name = name
        self.beam = beam
        self.source = source
        self.elements = optical_elements if optical_elements is not None else []
        self.beamline_frame = beamline_frame
        self.lab_frame = lab_frame
        self.construct_frames()

        self.beam.genSource(self.source.shadow_oe)

        self.beams = [self.beam.duplicate()]  # List to hold beams at each stage of the beamline

    def construct_frames(self):
        """
        Construct the reference frames for each optical element in the beamline.
        The source reference frame is defined relative to the beamline frame, 
        and each subsequent elements are defined relative to the previous element's frame.
        """
        if self.beamline_frame is None:
            raise ValueError("Beamline frame is not defined.")
        
        self.source.frame = self.beamline_frame.child_frame(relative_origin=[0, 0, 0], 
                                                            relative_rotation=np.eye(3),
                                                            name=f"{self.name}_{self.source.name}_frame")
        current_frame = self.source.frame

        # Iterate through the optical elements and construct their frames
        # Orientation is relative to the previous element, using the left-hand 
        # rule for positive angles
        for element in self.elements:
            relative_origin = [0, element.source_distance, 0]
            relative_orientation = rotation_matrix('y', -element.orientation)
            element.frame = current_frame.child_frame(relative_origin, 
                                                      relative_orientation,
                                                      name=f"{element.name}_frame")
            current_frame = element.frame

    def trace(self):
        """
        Trace the beam through the whole beamline.
        """
        
        for i, element in enumerate(self.elements):
            self.beam.traceOE(element.shadow_oe, i+1)
            
            # Store the beam after each element
            self.beams.append(self.beam.duplicate())

            lab_xhat = np.array([1, 0, 0])  #Lab x direction
            lab_yhat = np.array([0, 1, 0])  #Lab y direction

            local_x = element.frame.vector_from_lab(lab_xhat)
            local_y = element.frame.vector_from_lab(lab_yhat)

            x_index = np.argmax(np.abs(local_x))+1 # 1 = x, 2 = y, 3 = z
            y_index = np.argmax(np.abs(local_y))+1 # 1 = x, 2 = y, 3 = z

            element.image = self.beam.histo2(col_h=x_index, col_v=y_index, 
                                             nbins=200, nolost=1)
        



In [ ]:
source = BendingMagnet("B1", "source.props")

In [ ]:
len(source.shadow_oe.to_dictionary())

In [ ]:
beam = Shadow.Beam()
beam.genSource(source.shadow_oe)

In [ ]:
len(source.shadow_oe.to_dictionary())

In [ ]:
mirror = ToroidalMirror("M1", "mirror.props")
mirror.orientation
mirror.source_distance

In [ ]:
mirror.offset

In [ ]:
slit = Slits("Slit_A1", "slits.props")

In [ ]:
screen = Screen("DVF_B1", "screen.props")

In [ ]:
mirror.offset = []

In [ ]:
sirius_frame = ReferenceFrame(name="sirius")

In [ ]:
R1 = rotation_matrix('x', 90)
R2 = rotation_matrix('y', 180)
source_R = R1 @ R2
shadow_frame = sirius_frame.child_frame(relative_origin=[0, 0, 0], 
                                        relative_rotation=source_R, 
                                        name="shadow")

In [ ]:
optical_elements = [mirror, slit, screen]

In [ ]:
beam = Shadow.Beam()

In [ ]:
cax = BeamLine(name="CAX", source=source, optical_elements=optical_elements,
               beamline_frame=shadow_frame, lab_frame=sirius_frame, beam=beam)

In [ ]:
cax.trace()

In [ ]:
screen.image

In [ ]:
import matplotlib.pyplot as plt

ticket = screen.image

plt.pcolormesh(ticket["bin_h_edges"], ticket["bin_v_edges"], ticket["histogram"].T)
plt.xlabel("Horizontal position (mm)")
plt.ylabel("Vertical position (mm)")
plt.title("Beam profile at screen")
plt.colorbar(label="Intensity (a.u.)")
plt.axis('equal')
plt.show()

In [ ]:
mirror.specification

In [ ]:
Shadow.ShadowTools.plotxy(cax.beams[0],1,3,nbins=200,nolost=1,title="Real space")


In [ ]:
Shadow.ShadowTools.plotxy(cax.beams[3],1,3,nbins=101,nolost=1,title="Real space")


In [ ]:
cax.beam.retrace(12641)

In [ ]:
Shadow.ShadowTools.plotxy(cax.beam,1,3,nbins=101,nolost=1,title="Real space")


In [ ]:
cax.elements[1].frame

In [ ]:
mirror.offset

In [ ]:
mirror.offset = [0.1, 0.2, 0.3]

In [ ]:
source_x_vec = np.array([1, 0, 0])
source_y_vec = np.array([0, 1, 0])
source_z_vec = np.array([0, 0, 1])


In [ ]:
shadow_source_frame.vector_to_lab(source_z_vec)

In [ ]:
mirror_R = rot_matrix('y', -np.pi/2)

In [ ]:
shadow_mirror_frame = shadow_source_frame.child_frame(relative_origin=[0, 17000, 0],
                                                      relative_rotation=mirror_R,
                                                      name="shadow_mirror")

In [ ]:
mirror_x_vec = np.eye(3)[0]
mirror_y_vec = np.eye(3)[1]
mirror_z_vec = np.eye(3)[2]


In [ ]:
print(f"Mirror X vector in lab frame: {np.round(shadow_mirror_frame.vector_to_lab(mirror_x_vec))}")
print(f"Mirror Y vector in lab frame: {np.round(shadow_mirror_frame.vector_to_lab(mirror_y_vec))}")
print(f"Mirror Z vector in lab frame: {np.round(shadow_mirror_frame.vector_to_lab(mirror_z_vec))}")

In [ ]:
shadow_screens_frame = shadow_mirror_frame.child_frame(relative_origin=[0, 4359, 0],
                                                      relative_rotation=np.eye(3),
                                                      name="shadow_screen")

In [ ]:
np.round(shadow_screens_frame.point_to_lab(shadow_screens_frame.origin)[2]+12641)